# 작업대 물체 분류 모델을 GPU에서 파인튜닝하기

**목표:** Transformers가 불러온 모델의 분류 헤드를 학습한 뒤, 마지막 Transformer 블록도 조금 수정해 성능을 비교합니다.
PyTorch의 `DataLoader`, `CrossEntropyLoss`, `Adam`과 학습 반복문을 직접 읽습니다.
JAX 모델 내부 구현과 TPU 비교는 기존 심화 노트북에서 이어갑니다.

GPU 사용 여부, 검증 데이터로 선택한 모델, 고정한 가중치의 변화, 저장한 모델의 재로딩까지 확인합니다.
정확도 상승은 보장하지 않습니다. 데이터와 실행 환경에 따라 달라지는 실제 결과를 기록하세요.

**AI 코딩 실습:** 핵심 함수 두 개를 빈 코드 셀에 작성합니다. 문제 조건과 프롬프트를 읽고
코드를 만들어 작은 입력과 범위 변경으로 시도한 뒤, 같은 노트북에서 결과와 접힌 해설을 확인하세요.
별도 강사용 파일은 없습니다. 먼저 Codespaces에서 두 함수를 작성·저장하고 사전 검사를 통과한 뒤 Colab CLI로 실행합니다.
Codespaces CPU의 Run All은 이 GPU 노트북을 실행하지 못합니다. 빈칸이나 확인 실패가 있으면 본학습과 결과 저장을 중단합니다.
완료 보고서가 있는 폴더는 덮어쓰지 않습니다. 본학습을 다시 실행할 때는 기존 결과를 별도로 보관하세요.

In [ ]:
TRAINING_ALLOWED = False
EXERCISE_CHECKS = {"02-step": False, "02-scope": False}
HEAD_STAGE_COMPLETE = False
FINETUNE_SETUP_COMPLETE = False
FINETUNE_STAGE_COMPLETE = False
MODEL_RELOADED = False
TABLES_SAVED = False
REPORT_READY = False
for name in ("train_one_batch", "select_finetune_parameters", "report"):
    globals().pop(name, None)

## 1. 라이브러리와 GPU 확인

이 노트북은 **Colab CLI로 연결한 GPU 세션**에서 실행합니다. Codespaces는 파일 작성·업로드에 사용합니다.
`00_hf_download_and_data.ipynb`에서 받은 모델 파일과 기존 `data/prepared`의 데이터를 먼저 업로드하세요.
GPU가 없으면 여기서 멈춥니다. 설치와 세션 연결 순서는 이 폴더의 README를 따릅니다.

In [ ]:
TRAINING_ALLOWED = False
import os
import sys
import json
import hashlib
import random
from pathlib import Path
from importlib.metadata import version

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython import get_ipython
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")
if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. Colab CLI GPU 세션으로 실행하세요.")
device = torch.device("cuda")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("Python:", sys.executable)
print("GPU:", torch.cuda.get_device_name(0))
print({name: version(name) for name in ("torch", "transformers", "huggingface-hub")})

In [ ]:
candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/vision-ai")]
ROOT = next((p for p in candidates if (p / ".vision-lab-root").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(".vision-lab-root와 실습 파일을 먼저 업로드하세요.")
MODEL_ID = 'facebook/deit-tiny-patch16-224'
MODEL_REVISION = 'b3428f18dcc7b543470d07f14b4a4157815d1880'
MODEL_DIR = ROOT / "hf_colab_gpu/models/deit-tiny"
DATA_DIR = ROOT / "data/prepared"
OUTPUT_DIR = ROOT / "hf_colab_gpu/results/gpu"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

download_manifest = json.loads((MODEL_DIR / "download_manifest.json").read_text())
assert download_manifest["model_id"] == MODEL_ID
assert download_manifest["revision"] == MODEL_REVISION
for name in ("config.json", "preprocessor_config.json", "pytorch_model.bin"):
    assert sha256_file(MODEL_DIR / name) == download_manifest["files"][name], name
print("HF CLI 다운로드 파일 확인:", MODEL_ID, MODEL_REVISION[:12])

## 2. 데이터와 분할 확인

CIFAR-100에서 bottle·bowl·can·cup·plate를 골랐습니다. 원본은 32×32 이미지이며 모델 입력 크기로 확대합니다.
실제 작업대 사진과는 차이가 있으므로 이 결과를 현장 정확도로 해석하지 않습니다.
학습 500장, 검증 100장, 최종 평가 200장으로 나눴고, 학습 중 모델 선택에는 검증 데이터만 사용합니다.

In [ ]:
manifest_file = DATA_DIR / "manifest.json"
manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
classes = manifest["classes"]
if len(classes) < 2 or len(set(classes)) != len(classes):
    raise ValueError("클래스 목록을 확인하세요.")
identity = {k: manifest[k] for k in ("classes", "splits", "seed", "preprocess")}
fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
assert fingerprint == manifest["dataset_sha256"]
splits, seen_ids = {}, set()
for split in ("train", "validation", "test"):
    record = manifest["splits"][split]
    assert record["file"] == f"{split}.npz"
    file = DATA_DIR / record["file"]
    assert sha256_file(file) == record["sha256"], split
    with np.load(file, allow_pickle=False) as stored:
        images, labels, ids = stored["images"], stored["labels"], stored["ids"]
    assert images.dtype == np.uint8 and images.ndim == 4 and images.shape[-1] == 3
    assert labels.ndim == 1 and np.issubdtype(labels.dtype, np.integer)
    assert len(images) == len(labels) == len(ids) == record["count"] > 0
    assert labels.min() >= 0 and labels.max() < len(classes)
    assert len(set(ids)) == len(ids) and not seen_ids.intersection(ids)
    seen_ids.update(ids)
    splits[split] = {"images": images, "labels": labels, "ids": ids}
    print(split, len(labels), "images")
print("클래스 순서:", classes)

## 3. Transformers로 모델 불러오기

`AutoImageProcessor`가 크기 조정·정규화를 맡고, `AutoModelForImageClassification`이 사전학습 모델을 만듭니다.
`hf download`로 받은 로컬 파일만 사용하므로 이 단계에서 추가 다운로드는 하지 않습니다.
원본 가중치는 `.bin`이며 `weights_only=True`로 읽습니다. 실습 후 저장할 모델은 Safetensors 형식입니다.

In [ ]:
processor = AutoImageProcessor.from_pretrained(
    MODEL_DIR, local_files_only=True, use_fast=False,
)
model = AutoModelForImageClassification.from_pretrained(
    MODEL_DIR, local_files_only=True, use_safetensors=False, weights_only=True,
    attn_implementation="eager",
).to(device)
print(type(model).__name__, "· 원래 분류 수:", model.config.num_labels)
print("전체 파라미터:", f"{sum(p.numel() for p in model.parameters()):,}")

## 4. 학습 설정과 미니배치 만들기

배치 크기는 16, 학습은 단계별 3 epoch입니다. 이미지 전처리는 Transformers에 맡깁니다.
파일을 덮어쓰지 않도록 이전 `report.json`이 있으면 멈춥니다. 재실행할 때는 이전 결과 폴더를 보관한 뒤 새로 시작하세요.

In [ ]:
TRAINING_ALLOWED = False
if (OUTPUT_DIR / "report.json").exists():
    raise FileExistsError("완료된 결과가 있습니다. results/gpu를 보관한 뒤 재실행하세요.")
import csv
import time
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 16
HEAD_EPOCHS, FINETUNE_EPOCHS = 3, 3
HEAD_LR, FINETUNE_LR = 1e-3, 1e-4
started = time.perf_counter()
loaders = {}
for name, split in splits.items():
    pil_images = [Image.fromarray(pixels).convert("RGB") for pixels in split["images"]]
    pixels = processor(images=pil_images, return_tensors="pt")["pixel_values"]
    dataset = TensorDataset(pixels, torch.tensor(split["labels"], dtype=torch.long))
    loaders[name] = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=(name == "train"),
        generator=torch.Generator().manual_seed(SEED), num_workers=0,
    )
print("입력 크기:", tuple(loaders["train"].dataset.tensors[0].shape))
TRAINING_ALLOWED = True

## 5. 분류 헤드만 학습하도록 설정하기

1,000개 클래스를 출력하던 마지막 선형층을 실습 클래스 수에 맞춰 교체합니다.
나머지 가중치는 `requires_grad=False`로 고정합니다. 모델의 전체 구조를 다시 구현할 필요가 없습니다.

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
for parameter in model.parameters():
    parameter.requires_grad = False
model.classifier = nn.Linear(model.config.hidden_size, len(classes)).to(device)
nn.init.normal_(model.classifier.weight, std=model.config.initializer_range)
nn.init.zeros_(model.classifier.bias)
model.num_labels = len(classes)
model.config.num_labels = len(classes)
model.config.id2label = dict(enumerate(classes))
model.config.label2id = {label: index for index, label in enumerate(classes)}
criterion = nn.CrossEntropyLoss()

def parameter_sha256(named_parameters):
    digest = hashlib.sha256()
    for name, parameter in named_parameters:
        digest.update(name.encode())
        digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

backbone_before = parameter_sha256(model.vit.named_parameters())
history = []
print("헤드 학습 파라미터:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 6. 평가 함수 만들기

이 함수는 손실·정확도·예측값을 돌려줍니다. 학습 중에는 검증 데이터만 넣습니다.
최종 평가 데이터는 각 단계의 최적 모델을 고른 뒤에 사용합니다.

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
def evaluate(data_loader):
    model.eval()
    loss_sum, all_labels, all_predictions = 0.0, [], []
    with torch.inference_mode():
        for pixels, labels in data_loader:
            pixels, labels = pixels.to(device), labels.to(device)
            logits = model(pixel_values=pixels).logits
            loss_sum += criterion(logits, labels).item() * len(labels)
            all_labels.extend(labels.cpu().tolist())
            all_predictions.extend(logits.argmax(dim=-1).cpu().tolist())
    labels = np.asarray(all_labels)
    predictions = np.asarray(all_predictions)
    matrix = np.zeros((len(classes), len(classes)), dtype=int)
    np.add.at(matrix, (labels, predictions), 1)
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    f1 = np.divide(2 * matrix.diagonal(), denominator,
                   out=np.zeros(len(classes)), where=denominator > 0)
    return {
        "loss": loss_sum / len(labels), "accuracy": float(np.mean(labels == predictions)),
        "macro_f1": float(f1.mean()), "count": len(labels),
        "confusion_matrix": matrix.tolist(), "predictions": predictions.tolist(),
    }

## 7. 실습 1 · 한 배치의 학습을 AI와 구현하기

손실을 계산하는 데서 끝내지 않고 모델의 가중치를 실제로 한 번 바꾸는 함수를 만듭니다.
먼저 작은 연습 모델로 확인하고, 통과한 **같은 함수**를 아래 실제 DeiT 학습 반복문에서 사용합니다.

**제공 입력:** `model`, `optimizer`, `criterion`, 장치에 올라간 `pixels`와 `labels`입니다.
모델은 `model(pixel_values=pixels).logits`로 점수를 반환합니다.

**완성 조건**

1. `train_one_batch(model, optimizer, criterion, pixels, labels)` 함수를 만듭니다.
2. 학습 모드로 바꾸고 이전 기울기를 지운 다음, 예측 점수와 손실을 계산합니다.
3. 역전파와 optimizer 업데이트를 각각 한 번 실행합니다.
4. `(손실 float, 맞힌 수 int, 사진 수 int)` 튜플을 반환합니다.
   손실과 맞힌 수는 **이번 업데이트 전**에 구한 예측 점수로 계산합니다.
5. 새 optimizer나 모델을 만들지 않습니다. 제공된 장치·학습률·학습 대상 가중치를 유지합니다.

**연습 입력의 예상 결과:** 아래 2개 입력에서 첫 손실은 약 `0.6931`, 정답 수는 `1`,
전체 수는 `2`입니다. 한 번 학습한 뒤에는 가중치가 달라집니다. 같은 입력으로 다시 학습하면
이 작은 예제의 손실은 약 `0.6685`가 됩니다. 실제 사진 학습의 손실이나 정확도를 보장하는 값은 아닙니다.

**AI에게 요청하기**

> 저는 Python 초보자입니다. 위 입력과 완성 조건을 지키는 `train_one_batch` 함수를 작성해 주세요.
> PyTorch의 train, zero_grad, forward, CrossEntropyLoss, backward, step을 사용해 주세요.
> criterion은 이미 만들어져 있으며 반환값은 loss.item()의 float와 정답 개수, 배치 개수입니다.
> 업데이트 전 logits의 argmax로 맞힌 수를 구하고, 가중치를 직접 대입하거나 예시 정답을 고정하지 마세요.
> 기울기를 매번 지우는 이유와, backward와 step의 차이도 쉬운 말로 설명해 주세요.

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
from types import SimpleNamespace
EXERCISE_CHECKS["02-step"] = False

class PracticeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = nn.Linear(2, 2)

    def forward(self, pixel_values):
        return SimpleNamespace(logits=self.classifier(pixel_values))

def make_practice_model():
    # 연습 모델 초기화가 본학습의 난수 순서를 바꾸지 않도록 합니다.
    with torch.random.fork_rng(devices=[]):
        practice = PracticeClassifier().to(device)
    with torch.no_grad():
        practice.classifier.weight.zero_()
        practice.classifier.bias.zero_()
    return practice.eval()

practice_pixels = torch.tensor([[1.0, 0.0], [0.0, 1.0]], device=device)
practice_labels = torch.tensor([0, 1], dtype=torch.long, device=device)
print("연습 입력:", practice_pixels.tolist(), "· 정답:", practice_labels.tolist())
print("연습 optimizer: SGD, 학습률 0.1 / 실제 본학습: 기존 Adam 설정 유지")

아래 빈 코드 셀에 함수만 작성하세요. 실행 결과는 다음 확인 셀에서 봅니다.

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
EXERCISE_CHECKS["02-step"] = False
if not callable(globals().get("train_one_batch")):
    raise RuntimeError("실습 1의 빈 셀에 train_one_batch 함수를 만들고 실행하세요.")
practice = make_practice_model()
practice_optimizer = torch.optim.SGD(practice.parameters(), lr=0.1)
first = train_one_batch(practice, practice_optimizer, criterion, practice_pixels, practice_labels)
second = train_one_batch(practice, practice_optimizer, criterion, practice_pixels, practice_labels)
for result in (first, second):
    assert isinstance(result, tuple) and len(result) == 3, "튜플 (손실, 맞힌 수, 사진 수)을 반환하세요."
    assert type(result[0]) is float and all(type(x) is int for x in result[1:])
assert practice.training, "함수 안에서 학습 모드로 바꾸세요."
assert abs(first[0] - 0.69314718) < 1e-5 and first[1:] == (1, 2)
assert abs(second[0] - 0.66845965) < 1e-5 and second[1:] == (2, 2)
expected_weight = torch.tensor([[0.04937513, -0.04937513], [-0.04937513, 0.04937513]], device=device)
torch.testing.assert_close(practice.classifier.weight, expected_weight, rtol=1e-5, atol=1e-6)
torch.testing.assert_close(practice.classifier.bias, torch.zeros(2, device=device), rtol=0, atol=1e-6)
EXERCISE_CHECKS["02-step"] = True
print(f"1회: loss={first[0]:.4f}, correct={first[1]}, count={first[2]}")
print(f"2회: loss={second[0]:.4f}, correct={second[1]}, count={second[2]}")
print("실습 1 확인 통과: 두 번의 업데이트와 기울기 초기화 결과가 맞습니다.")

### 다른 학습률로 시도하기

아래는 **작은 연습 모델만** 새로 만들어 학습률 `0.01`과 `0.1`로 두 번씩 학습합니다.
어느 쪽의 두 번째 손실이 더 작을지 예상해 보세요. 본학습 모델이나 정해 둔 학습률은 바꾸지 않습니다.

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not EXERCISE_CHECKS.get("02-step", False):
    raise RuntimeError("실습 1의 확인 셀을 먼저 통과하세요.")
for practice_lr in (0.01, 0.1):
    trial_model = make_practice_model()
    trial_optimizer = torch.optim.SGD(trial_model.parameters(), lr=practice_lr)
    losses = []
    for _ in range(2):
        result = train_one_batch(trial_model, trial_optimizer, criterion, practice_pixels, practice_labels)
        losses.append(result[0])
    print(f"연습 학습률 {practice_lr}: {losses[0]:.4f} → {losses[1]:.4f}")

<details>
<summary>시도한 뒤 결과 해설 보기</summary>

backward는 각 가중치를 어느 방향으로 바꿀지 기울기를 계산하고, step은 optimizer가 그 기울기를 사용해
가중치를 바꾸는 단계입니다. 다음 배치를 학습할 때 zero_grad를 빠뜨리면 이전 기울기가 누적됩니다.
이 작은 예제에서는 0.1이 더 빠르게 손실을 낮춥니다. 모든 모델에서 학습률이 클수록 좋다는 뜻은 아닙니다.
아래 실제 이미지 학습은 원래 설정인 Adam과 학습률 0.001을 사용합니다.

</details>

### 완성한 함수로 분류 헤드 학습하기

검증 정확도가 가장 높은 epoch를 선택하고 동률이면 검증 손실이 낮은 모델을 고릅니다.
테스트 결과로 epoch를 선택하지 않습니다. 이제부터 아래 결과는 실제 GPU 학습 결과입니다.

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not EXERCISE_CHECKS.get("02-step", False):
    raise RuntimeError("실습 1의 확인 셀을 먼저 통과하세요.")
HEAD_STAGE_COMPLETE = False
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=HEAD_LR)
best_key, best_head, best_head_epoch = (-1.0, float("-inf")), None, None
for epoch in range(1, HEAD_EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_count = 0.0, 0, 0
    for pixels, labels in loaders["train"]:
        pixels, labels = pixels.to(device), labels.to(device)
        loss_value, correct_count, batch_count = train_one_batch(model, optimizer, criterion, pixels, labels)
        train_loss += loss_value * batch_count
        train_correct += correct_count
        train_count += batch_count
    validation = evaluate(loaders["validation"])
    history.append({"stage": "head", "epoch": epoch, "train_loss": train_loss / train_count,
                    "train_accuracy": train_correct / train_count,
                    "validation_loss": validation["loss"], "validation_accuracy": validation["accuracy"]})
    key = (validation["accuracy"], -validation["loss"])
    if key > best_key:
        best_key, best_head_epoch = key, epoch
        best_head = {name: value.detach().cpu().clone()
                     for name, value in model.classifier.state_dict().items()}
    print(f"head {epoch}/{HEAD_EPOCHS} · validation {validation['accuracy']:.1%}")
model.classifier.load_state_dict(best_head)
assert parameter_sha256(model.vit.named_parameters()) == backbone_before
head_validation = evaluate(loaders["validation"])
head_test = evaluate(loaders["test"])
print(f"선택 epoch {best_head_epoch} · head test {head_test['accuracy']:.1%}")
HEAD_STAGE_COMPLETE = True

## 8. 실습 2 · 학습할 가중치 범위 선택하기

방금 학습한 분류 헤드를 유지한 채, 어느 부분까지 추가로 학습할지 정하는 함수를 만듭니다.

**제공 입력:** 분류 헤드 학습을 마친 `model`과 문자열 `scope`입니다.
이번 DeiT에는 `model.vit.encoder.layer`, `model.vit.layernorm`, `model.classifier`가 있습니다.

**완성 조건**

1. `select_finetune_parameters(model, scope)` 함수를 만듭니다.
2. 호출할 때마다 모든 가중치의 `requires_grad`를 먼저 False로 되돌립니다.
3. `scope="head"`면 classifier만, `scope="last_block"`이면 마지막 Transformer 블록·최종 layernorm·classifier만 True로 바꿉니다.
4. 학습할 파라미터만 `{이름: 실제 parameter 객체}` 사전으로 반환합니다.
   이름은 `model.named_parameters()`에서 얻고 새 텐서나 모델을 만들지 않습니다.
5. 두 범위 이외의 문자열은 ValueError를 내고, 가중치 값 자체는 바꾸지 않습니다.

**예상 결과:** 현재 모델과 5개 클래스에서는 head만 `965`개, last_block 범위는 `446,213`개입니다.
모델이나 클래스 수가 달라지면 개수도 달라집니다. 이 숫자를 반환값으로 고정하지 마세요.

**AI에게 요청하기**

> 위 구조와 조건에 맞는 `select_finetune_parameters(model, scope)` 함수를 작성해 주세요.
> head와 last_block 범위를 모두 지원하고 마지막 블록 번호는 layer[-1]로 찾아 주세요.
> 범위를 바꿔 다시 호출해도 전에 True였던 가중치가 남지 않도록 해 주세요.
> 이름과 실제 parameter 객체가 담긴 사전을 반환하고 파일 저장·학습·가중치 재초기화는 하지 마세요.
> requires_grad와 파라미터 개수가 무엇을 의미하는지 초보자에게 설명해 주세요.

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
EXERCISE_CHECKS["02-scope"] = False
FINETUNE_SETUP_COMPLETE = False
if not HEAD_STAGE_COMPLETE:
    raise RuntimeError("분류 헤드 학습과 평가를 먼저 완료하세요.")
print("완료한 분류 헤드의 검증 정확도:", f"{head_validation['accuracy']:.1%}")
print("Transformer 블록 수:", len(model.vit.encoder.layer))
print("분류할 종류:", model.config.num_labels)

아래 빈 코드 셀을 채우세요. 다음 셀에서 범위를 바꾸며 개수와 고정 여부를 확인합니다.

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
EXERCISE_CHECKS["02-scope"] = False
if not HEAD_STAGE_COMPLETE:
    raise RuntimeError("분류 헤드 학습을 먼저 완료하세요.")
if not callable(globals().get("select_finetune_parameters")):
    raise RuntimeError("실습 2의 빈 셀에 select_finetune_parameters 함수를 작성하세요.")
unchanged_weights = parameter_sha256(model.named_parameters())
last_prefix = f"vit.encoder.layer.{len(model.vit.encoder.layer) - 1}."
parameter_lookup = dict(model.named_parameters())
for scope in ("head", "last_block", "head", "last_block"):
    chosen = select_finetune_parameters(model, scope)
    prefixes = ("classifier.",) if scope == "head" else (last_prefix, "vit.layernorm.", "classifier.")
    expected = {name for name in parameter_lookup if name.startswith(prefixes)}
    assert isinstance(chosen, dict) and set(chosen) == expected, "선택한 가중치의 이름을 확인하세요."
    assert all(chosen[name] is parameter_lookup[name] for name in expected), "원래 parameter 객체를 반환하세요."
    assert {name for name, p in model.named_parameters() if p.requires_grad} == expected
    assert parameter_sha256(model.named_parameters()) == unchanged_weights, "가중치 값은 바꾸지 마세요."
    print(f"범위 {scope}: {sum(p.numel() for p in chosen.values()):,}개 파라미터")
try:
    select_finetune_parameters(model, "unknown")
except ValueError:
    pass
else:
    raise AssertionError("지원하지 않는 범위에는 ValueError가 필요합니다.")
# 위 오류 입력까지 확인한 뒤 본학습 범위를 다시 적용합니다.
chosen = select_finetune_parameters(model, "last_block")
assert set(chosen) == expected
assert {n for n, p in model.named_parameters() if p.requires_grad} == expected
assert all(chosen[name] is parameter_lookup[name] for name in expected)
assert parameter_sha256(model.named_parameters()) == unchanged_weights
EXERCISE_CHECKS["02-scope"] = True
print("실습 2 확인 통과: 범위를 다시 바꿔도 고정 상태와 원래 가중치를 유지했습니다.")

### 범위를 바꾼 결과 설명하기

방금 셀은 head → last_block → head → last_block 순서로 호출했습니다.
세 번째 출력이 다시 965가 되는 이유와, 마지막에 last_block으로 돌려놓은 이유를 자신의 말로 설명해 보세요.

<details>
<summary>시도한 뒤 결과 해설 보기</summary>

requires_grad는 이번 학습에서 기울기를 계산할 가중치를 고르는 설정입니다.
범위를 바꿀 때 먼저 전부 고정해야 이전 선택이 남지 않습니다. 학습할 수 있는 가중치가 많아졌다는 사실만으로
정확도가 높아진다고 판단할 수 없습니다. 아래 실제 학습 뒤 같은 테스트 이미지에서 비교합니다.
실습의 본학습은 마지막 블록·최종 정규화·분류 헤드로 범위를 맞추고 학습률 0.0001을 사용합니다.

</details>

### 선택한 범위로 실제 파인튜닝하기

앞의 11개 블록을 고정한 채 방금 학습한 분류 헤드에서 이어서 시작합니다.
손실 계산과 업데이트에는 실습 1에서 만든 함수를 그대로 사용합니다.

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
FINETUNE_SETUP_COMPLETE = False
if not HEAD_STAGE_COMPLETE or not all(EXERCISE_CHECKS.values()):
    raise RuntimeError("두 실습 확인과 분류 헤드 학습을 먼저 완료하세요.")
last_block_index = len(model.vit.encoder.layer) - 1
last_block_prefix = f"vit.encoder.layer.{last_block_index}."
trainable = select_finetune_parameters(model, "last_block")
frozen_before = parameter_sha256((n, p) for n, p in model.named_parameters() if not p.requires_grad)
tail_before = parameter_sha256(model.vit.encoder.layer[-1].named_parameters())
trainable_count = sum(parameter.numel() for parameter in trainable.values())
assert all(name.startswith((last_block_prefix, "vit.layernorm.", "classifier.")) for name in trainable)
print("파인튜닝 파라미터:", f"{trainable_count:,}")
optimizer = torch.optim.Adam(trainable.values(), lr=FINETUNE_LR)
best_key, best_tail, best_finetune_epoch = (-1.0, float("-inf")), None, None
FINETUNE_SETUP_COMPLETE = True

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
FINETUNE_STAGE_COMPLETE = False
if not FINETUNE_SETUP_COMPLETE or not all(EXERCISE_CHECKS.values()):
    raise RuntimeError("두 실습 확인과 파인튜닝 준비를 먼저 완료하세요.")
for epoch in range(1, FINETUNE_EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_count = 0.0, 0, 0
    for pixels, labels in loaders["train"]:
        pixels, labels = pixels.to(device), labels.to(device)
        loss_value, correct_count, batch_count = train_one_batch(model, optimizer, criterion, pixels, labels)
        train_loss += loss_value * batch_count
        train_correct += correct_count
        train_count += batch_count
    validation = evaluate(loaders["validation"])
    history.append({"stage": "finetune", "epoch": epoch, "train_loss": train_loss / train_count,
                    "train_accuracy": train_correct / train_count,
                    "validation_loss": validation["loss"], "validation_accuracy": validation["accuracy"]})
    key = (validation["accuracy"], -validation["loss"])
    if key > best_key:
        best_key, best_finetune_epoch = key, epoch
        best_tail = {name: parameter.detach().cpu().clone() for name, parameter in trainable.items()}
    print(f"finetune {epoch}/{FINETUNE_EPOCHS} · validation {validation['accuracy']:.1%}")
with torch.no_grad():
    for name, parameter in trainable.items():
        parameter.copy_(best_tail[name].to(device))
finetune_validation = evaluate(loaders["validation"])
finetune_test = evaluate(loaders["test"])
frozen_after = parameter_sha256((n, p) for n, p in model.named_parameters() if not p.requires_grad)
tail_after = parameter_sha256(model.vit.encoder.layer[-1].named_parameters())
assert frozen_before == frozen_after, "고정한 가중치가 바뀌었습니다."
assert tail_before != tail_after, "마지막 블록의 가중치가 바뀌지 않았습니다."
print(f"선택 epoch {best_finetune_epoch} · fine-tune test {finetune_test['accuracy']:.1%}")
FINETUNE_STAGE_COMPLETE = True

## 9. 학습 전후 비교하기

Head는 분류 헤드만 학습한 기준 모델, Fine-tune은 마지막 블록까지 학습한 모델입니다.
학습 곡선에서는 검증 성능을, 혼동행렬에서는 마지막에 남겨 둔 평가 이미지의 오분류를 확인합니다.
두 단계 모두 같은 200장으로 평가하므로 단순히 숫자가 올랐는지뿐 아니라 어떤 클래스가 달라졌는지도 살펴보세요.

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not FINETUNE_STAGE_COMPLETE:
    raise RuntimeError("실습 확인과 실제 학습을 완료한 뒤 결과를 확인하세요.")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for stage, color in (("head", "#1a73e8"), ("finetune", "#188038")):
    rows = [row for row in history if row["stage"] == stage]
    epochs = [row["epoch"] for row in rows]
    axes[0].plot(epochs, [r["train_loss"] for r in rows], "o-", color=color, label=f"{stage}: train")
    axes[0].plot(epochs, [r["validation_loss"] for r in rows], "s--", color=color, label=f"{stage}: validation")
    axes[1].plot(epochs, [r["validation_accuracy"] for r in rows], "o-", color=color, label=stage)
axes[0].set(title="Loss by training stage", xlabel="Epoch within each stage", ylabel="Cross-entropy loss")
axes[1].set(title="Validation accuracy (100 images)", xlabel="Epoch within each stage", ylabel="Accuracy", ylim=(0, 1))
for axis in axes:
    axis.legend(fontsize=9)
    axis.grid(alpha=0.2)
    axis.set_xticks(range(1, max(HEAD_EPOCHS, FINETUNE_EPOCHS) + 1))
fig.savefig(OUTPUT_DIR / "learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not FINETUNE_STAGE_COMPLETE:
    raise RuntimeError("실습 확인과 실제 학습을 완료한 뒤 결과를 확인하세요.")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
maximum = max(np.max(head_test["confusion_matrix"]), np.max(finetune_test["confusion_matrix"]))
for axis, title, metrics in zip(axes, ("Head", "Fine-tune"), (head_test, finetune_test)):
    matrix = np.asarray(metrics["confusion_matrix"])
    image = axis.imshow(matrix, vmin=0, vmax=maximum, cmap="Blues")
    axis.set_xticks(range(len(classes)), classes, rotation=35, ha="right")
    axis.set_yticks(range(len(classes)), classes)
    axis.set(title=f"{title}: {metrics['accuracy']:.1%} ({metrics['count']} test images)",
             xlabel="Predicted class", ylabel="True class")
    for row, column in np.ndindex(matrix.shape):
        axis.text(column, row, str(matrix[row, column]), ha="center", va="center",
                  color="white" if matrix[row, column] > maximum / 2 else "black")
fig.colorbar(image, ax=axes, label="Image count", shrink=0.85)
fig.savefig(OUTPUT_DIR / "confusion_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"정확도 변화: {head_test['accuracy']:.1%} → {finetune_test['accuracy']:.1%}")
print(f"Macro F1 변화: {head_test['macro_f1']:.3f} → {finetune_test['macro_f1']:.3f}")

## 10. 모델 저장 후 다시 불러오기

`save_pretrained()`가 모델 설정과 Safetensors 가중치를 함께 저장합니다.
저장된 폴더를 `from_pretrained()`로 다시 불러와 같은 이미지의 출력이 일치하는지 확인합니다.

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not FINETUNE_STAGE_COMPLETE:
    raise RuntimeError("실습 확인과 실제 학습을 완료한 뒤 결과를 확인하세요.")
MODEL_RELOADED = False
saved_model_dir = OUTPUT_DIR / "finetuned_model"
model.save_pretrained(saved_model_dir, safe_serialization=True)
processor.save_pretrained(saved_model_dir)
restored_processor = AutoImageProcessor.from_pretrained(saved_model_dir, local_files_only=True, use_fast=False)
restored = AutoModelForImageClassification.from_pretrained(
    saved_model_dir, local_files_only=True, use_safetensors=True, attn_implementation="eager",
).to(device).eval()
check_image = Image.fromarray(splits["test"]["images"][0]).convert("RGB")
check_input = processor(images=check_image, return_tensors="pt").to(device)
restored_input = restored_processor(images=check_image, return_tensors="pt").to(device)
model.eval()
with torch.inference_mode():
    before_save = model(**check_input).logits
    after_load = restored(**restored_input).logits
torch.testing.assert_close(before_save, after_load, rtol=1e-5, atol=1e-6)
assert restored.config.id2label == dict(enumerate(classes))
reload_max_abs_diff = float((before_save - after_load).abs().max())
print("저장·재로딩 검증 통과 · 최대 출력 차이:", reload_max_abs_diff)
MODEL_RELOADED = True

## 11. 결과 파일과 실행 기록 저장

`report.json`은 실제 장치·데이터·학습 설정·평가 결과를 담습니다.
Colab 세션을 종료하기 전에 결과 폴더와 `finetuned_model`을 Codespaces로 내려받으세요.

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not FINETUNE_STAGE_COMPLETE:
    raise RuntimeError("실습 확인과 실제 학습을 완료한 뒤 결과를 확인하세요.")
if not MODEL_RELOADED:
    raise RuntimeError("모델 저장·재로딩 확인을 먼저 완료하세요.")
TABLES_SAVED = False
with (OUTPUT_DIR / "training.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(history[0]))
    writer.writeheader()
    writer.writerows(history)
with (OUTPUT_DIR / "predictions.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["sample_id", "true_label", "head_prediction", "finetune_prediction"])
    for sample_id, label, head_pred, fine_pred in zip(
        splits["test"]["ids"], splits["test"]["labels"], head_test["predictions"], finetune_test["predictions"],
    ):
        writer.writerow([sample_id, classes[int(label)], classes[head_pred], classes[fine_pred]])
def summary(metrics):
    return {key: value for key, value in metrics.items() if key != "predictions"}

TABLES_SAVED = True

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not FINETUNE_STAGE_COMPLETE:
    raise RuntimeError("실습 확인과 실제 학습을 완료한 뒤 결과를 확인하세요.")
if not MODEL_RELOADED:
    raise RuntimeError("모델 저장·재로딩 확인을 먼저 완료하세요.")
REPORT_READY = False
if not TABLES_SAVED:
    raise RuntimeError("결과 표 저장을 먼저 완료하세요.")
report = {
    "status": "completed", "platform": "gpu", "accelerator_verified": True,
    "device": torch.cuda.get_device_name(0), "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "download_manifest_sha256": sha256_file(MODEL_DIR / "download_manifest.json"),
    "dataset_manifest_sha256": sha256_file(manifest_file), "dataset_sha256": manifest["dataset_sha256"],
    "classes": classes, "split_counts": {k: len(v["labels"]) for k, v in splits.items()},
    "versions": {name: version(name) for name in ("torch", "transformers", "huggingface-hub")},
    "config": {"seed": SEED, "batch_size": BATCH_SIZE, "head_epochs": HEAD_EPOCHS,
               "finetune_epochs": FINETUNE_EPOCHS, "head_lr": HEAD_LR, "finetune_lr": FINETUNE_LR,
               "optimizer": "Adam", "checkpoint_selection": "validation_accuracy_then_loss"},
    "trainable_parameters": trainable_count,
    "stages": {"head": {"selected_epoch": best_head_epoch, "validation": summary(head_validation), "test": summary(head_test)},
               "finetune": {"selected_epoch": best_finetune_epoch, "validation": summary(finetune_validation), "test": summary(finetune_test)}},
    "verification": {"frozen_parameters_unchanged": frozen_before == frozen_after,
                     "last_block_changed": tail_before != tail_after, "reload_max_abs_diff": reload_max_abs_diff,
                     "frozen_before": frozen_before, "frozen_after": frozen_after,
                     "last_block_before": tail_before, "last_block_after": tail_after},
    "checkpoint": {"path": "finetuned_model", "format": "safetensors",
                   "sha256": sha256_file(saved_model_dir / "model.safetensors")},
    "elapsed_seconds": time.perf_counter() - started,
}
REPORT_READY = True

In [ ]:
if not TRAINING_ALLOWED:
    raise RuntimeError("학습 준비가 완료되지 않았습니다. 기존 결과를 보관한 뒤 처음부터 실행하세요.")
if not FINETUNE_STAGE_COMPLETE:
    raise RuntimeError("실습 확인과 실제 학습을 완료한 뒤 결과를 확인하세요.")
if not REPORT_READY:
    raise RuntimeError("현재 실행의 보고서 준비가 완료되지 않았습니다.")
artifact_names = [
    "training.csv", "predictions.csv", "learning_curves.png", "confusion_comparison.png",
    "finetuned_model/config.json", "finetuned_model/preprocessor_config.json",
    "finetuned_model/model.safetensors",
]
report["artifacts"] = artifact_names
report["artifact_sha256"] = {
    name: sha256_file(OUTPUT_DIR / name) for name in artifact_names
}
(OUTPUT_DIR / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("HF_GPU_FINETUNING_COMPLETE", OUTPUT_DIR / "report.json")

## 다음 실습

결과 표와 오분류 이미지를 보고 데이터 수·촬영 조건·학습 범위를 어떻게 바꿀지 정해 보세요.
저장된 `finetuned_model` 폴더는 `AutoImageProcessor`와 `AutoModelForImageClassification`로 바로 다시 읽을 수 있습니다.
JAX·TPU에 관심이 있다면 상위 폴더의 기존 심화 노트북에서 같은 과제를 다른 실행 환경과 비교해 보세요.